## [F] GMM

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from pydynpd import regression # Source: https://github.com/dazhwu/pydynpd

In [2]:
path = "data/processed_data/"
file_name = "03_pols_fe_re_gmm_winsorized_data.csv"

date_column = "date" 
stock_column = "stock"

# Set path
os.chdir("..")
print(os.getcwd())

/workspaces/master-thesis-submission/src


In [3]:
pols_fe_re_data = pd.read_csv(path + file_name)

In [4]:
pols_fe_re_data.head()

,date,stock,stock_return_quarterly,beta,size,value,profitability,investment,price_momentum,esg_score,esg_momentum,industry,country_of_headquarters
0,2016-06-30,A2.MI,0.030621,1.470557,22.009083,0.748281,0.079054,-0.033443,-0.036842,59.459100,-1.622974,Utilities,Italy
1,2016-06-30,AAL.L,0.250263,0.558792,23.225868,1.461129,-0.304424,0.000000,0.324333,81.434442,-7.452243,Basic Materials,United Kingdom
2,2016-06-30,AALB.AS,-0.112824,1.229154,21.818938,0.424145,0.194845,0.000000,0.014720,17.964003,5.607441,Industrials,Netherlands
3,2016-06-30,ABBN.S,0.029602,0.988284,24.365696,0.362603,0.200431,0.011075,0.041707,88.899243,-3.994408,Industrials,Switzerland
4,2016-06-30,ABDN.L,-0.217060,1.664075,22.662034,0.796699,0.150175,0.000000,-0.256856,74.424211,-0.391662,Financials,United Kingdom


## GMM Analysis

### 1 Data Preparation

In [5]:
gmm_data = pols_fe_re_data.copy()

# Convert date column to datetime and sort by stock and date
gmm_data["date"] = pd.to_datetime(gmm_data["date"])
gmm_data = gmm_data.sort_values(["stock", "date"])

In [6]:
# Determine consecutive quarters for each stock and assign run_id to identify consecutive runs
gmm_data["q_ord"] = gmm_data["date"].dt.to_period("Q").astype("int64")
is_break = gmm_data.groupby("stock")["q_ord"].diff().fillna(1).ne(1)
gmm_data["run_id"] = is_break.groupby(gmm_data["stock"]).cumsum()

max_run = gmm_data.groupby(["stock", "run_id"]).size().groupby("stock").max()
keep_stocks = max_run[max_run > 7].index
gmm_data = gmm_data[gmm_data["stock"].isin(keep_stocks)].copy()

gmm_data = gmm_data.drop(columns=["q_ord", "run_id"])

In [ ]:
gmm_data = gmm_data.sort_values(["stock", "date"])

#Robustness checks: split data into three parts and create time variable for each part
gmm_data_split_1 = gmm_data[gmm_data["date"] <= "2019-06-30"].copy()
gmm_data_split_2 = gmm_data[(gmm_data["date"] > "2019-06-30") & (gmm_data["date"] <= "2022-09-30")].copy()
gmm_data_split_3 = gmm_data[gmm_data["date"] > "2022-09-30"].copy()

gmm_data_split_1 = gmm_data_split_1.sort_values(["stock", "date"])
gmm_data_split_2 = gmm_data_split_2.sort_values(["stock", "date"])
gmm_data_split_3 = gmm_data_split_3.sort_values(["stock", "date"])

gmm_data["t"] = gmm_data["date"].astype("category").cat.codes + 1

gmm_data_split_1["t"] = gmm_data_split_1["date"].astype("category").cat.codes + 1
gmm_data_split_2["t"] = gmm_data_split_2["date"].astype("category").cat.codes + 1
gmm_data_split_3["t"] = gmm_data_split_3["date"].astype("category").cat.codes + 1

In [9]:
y = "stock_return_quarterly"

X = [
    "beta", "size", "value", "profitability", "investment",
    "price_momentum", "esg_score", "esg_momentum"
]

use = ["stock", "t", "date", y] + X

df_model = gmm_data[use].dropna().copy()
df_model_split_1 = gmm_data_split_1[use].dropna().copy()
df_model_split_2 = gmm_data_split_2[use].dropna().copy()
df_model_split_3 = gmm_data_split_3[use].dropna().copy()

### 2. GMM Analysis

#### GMM (1)

In [10]:
command_gmm_1 = """
stock_return_quarterly
L(1:1).stock_return_quarterly
beta 
size 
value 
profitability 
investment
price_momentum 
esg_score 
esg_momentum 
| gmm(stock_return_quarterly, 2:3)
  gmm(esg_score esg_momentum size beta value investment profitability, 3:3)
  gmm(price_momentum, 4:5)
| timedumm collapse nolevel fod
"""

results_gmm_1 = regression.abond(command_gmm_1, df_model, ["stock", "t"])
print(results_gmm_1)

 Dynamic panel-data estimation, two-step difference GMM
 Group variable: stock                            Number of obs = 17203    
 Time variable: t                                 Min obs per group: 4     
 Number of instruments = 45                       Max obs per group: 34    
 Number of groups = 672                           Avg obs per group: 25.60 
+---------------------------+------------+---------------------+-------------+-----------+-----+
|   stock_return_quarterly  |   coef.    | Corrected Std. Err. |      z      |   P>|z|   |     |
+---------------------------+------------+---------------------+-------------+-----------+-----+
| L1.stock_return_quarterly | 0.0166950  |      0.0346887      |  0.4812806  | 0.6303171 |     |
|            beta           | 0.0054467  |      0.0057977      |  0.9394485  | 0.3475005 |     |
|            size           | 0.0305511  |      0.0402346      |  0.7593242  | 0.4476586 |     |
|           value           | -0.0813478 |      0.0391102 

#### GMM (2)

In [11]:
command_gmm_2 = """
stock_return_quarterly
L(1:1).stock_return_quarterly
beta 
size 
value 
profitability 
investment
price_momentum 
esg_score 
esg_momentum 
| gmm(stock_return_quarterly, 2:3)
  gmm(esg_score esg_momentum size beta value investment profitability, 4:4)
  gmm(price_momentum, 6:6)
| timedumm collapse nolevel fod
"""

results_gmm_2 = regression.abond(command_gmm_2, df_model, ["stock", "t"])
print(results_gmm_2)

 Dynamic panel-data estimation, two-step difference GMM
 Group variable: stock                            Number of obs = 16244    
 Time variable: t                                 Min obs per group: 2     
 Number of instruments = 42                       Max obs per group: 32    
 Number of groups = 672                           Avg obs per group: 24.17 
+---------------------------+------------+---------------------+-------------+-----------+-----+
|   stock_return_quarterly  |   coef.    | Corrected Std. Err. |      z      |   P>|z|   |     |
+---------------------------+------------+---------------------+-------------+-----------+-----+
| L1.stock_return_quarterly | 0.0114027  |      0.0324623      |  0.3512607  | 0.7253927 |     |
|            beta           | 0.0273450  |      0.0085981      |  3.1803530  | 0.0014710 |  ** |
|            size           | 0.0079718  |      0.0428330      |  0.1861130  | 0.8523562 |     |
|           value           | -0.0808913 |      0.0381822 

#### GMM (3)

In [12]:
command_gmm_3 = """
stock_return_quarterly
L(1:1).stock_return_quarterly
beta 
size 
value 
profitability 
investment
price_momentum 
esg_score 
esg_momentum 
| gmm(stock_return_quarterly, 2:3)
  gmm(esg_score esg_momentum size beta value investment profitability price_momentum, 3:3)
| timedumm collapse nolevel fod
"""

results_gmm_3 = regression.abond(command_gmm_3, df_model, ["stock", "t"])
print(results_gmm_3)

 Dynamic panel-data estimation, two-step difference GMM
 Group variable: stock                            Number of obs = 17661    
 Time variable: t                                 Min obs per group: 5     
 Number of instruments = 45                       Max obs per group: 35    
 Number of groups = 672                           Avg obs per group: 26.28 
+---------------------------+------------+---------------------+-------------+-----------+-----+
|   stock_return_quarterly  |   coef.    | Corrected Std. Err. |      z      |   P>|z|   |     |
+---------------------------+------------+---------------------+-------------+-----------+-----+
| L1.stock_return_quarterly | 0.0164703  |      0.0339148      |  0.4856366  | 0.6272248 |     |
|            beta           | 0.0084594  |      0.0057886      |  1.4613847  | 0.1439099 |     |
|            size           | 0.0401971  |      0.0388273      |  1.0352797  | 0.3005383 |     |
|           value           | -0.0498663 |      0.0378562 

#### GMM (4)

In [13]:
command_gmm_4 = """
stock_return_quarterly
L(1:1).stock_return_quarterly
beta 
size 
value 
profitability 
investment
price_momentum 
esg_score 
esg_momentum 
| gmm(stock_return_quarterly, 2:3)
  gmm(esg_score esg_momentum size beta value investment price_momentum, 3:3)
  iv(profitability)
| timedumm collapse nolevel fod
"""

results_gmm_4 = regression.abond(command_gmm_4, df_model, ["stock", "t"])
print(results_gmm_4)

 Dynamic panel-data estimation, two-step difference GMM
 Group variable: stock                            Number of obs = 17661    
 Time variable: t                                 Min obs per group: 5     
 Number of instruments = 45                       Max obs per group: 35    
 Number of groups = 672                           Avg obs per group: 26.28 
+---------------------------+------------+---------------------+-------------+-----------+-----+
|   stock_return_quarterly  |   coef.    | Corrected Std. Err. |      z      |   P>|z|   |     |
+---------------------------+------------+---------------------+-------------+-----------+-----+
| L1.stock_return_quarterly | 0.0215555  |      0.0344775      |  0.6252049  | 0.5318366 |     |
|            beta           | 0.0065795  |      0.0059284      |  1.1098197  | 0.2670767 |     |
|            size           | 0.0510048  |      0.0404492      |  1.2609620  | 0.2073226 |     |
|           value           | -0.0569153 |      0.0397913 

#### GMM (5)

In [14]:
command_gmm_5 = """
stock_return_quarterly 
L(1:1).stock_return_quarterly
beta 
size 
value 
profitability 
investment
price_momentum 
esg_score 
esg_momentum 
| gmm(stock_return_quarterly, 2:3)
  gmm(esg_score esg_momentum value size profitability beta investment, 3:3)
  gmm(price_momentum, 4:5)
| timedumm collapse nolevel fod
"""

results_gmm_5 = regression.abond(command_gmm_5, df_model_split_1, ["stock", "t"])
print(results_gmm_5)

 Dynamic panel-data estimation, two-step difference GMM
 Group variable: stock                            Number of obs = 4405    
 Time variable: t                                 Min obs per group: 0    
 Number of instruments = 20                       Max obs per group: 9    
 Number of groups = 565                           Avg obs per group: 7.80 
+---------------------------+------------+---------------------+------------+-----------+-----+
|   stock_return_quarterly  |   coef.    | Corrected Std. Err. |     z      |   P>|z|   |     |
+---------------------------+------------+---------------------+------------+-----------+-----+
| L1.stock_return_quarterly | 0.0147011  |      0.1140719      | 0.1288755  | 0.8974562 |     |
|            beta           | 0.0052098  |      0.0218092      | 0.2388827  | 0.8111965 |     |
|            size           | 0.0753515  |      0.1942238      | 0.3879622  | 0.6980440 |     |
|           value           | -0.1913232 |      0.2838569      | -0.

#### GMM (6)

In [15]:
command_gmm_6 = """
stock_return_quarterly 
L(1:1).stock_return_quarterly
beta 
size 
value 
profitability 
investment
price_momentum 
esg_score 
esg_momentum 
| gmm(stock_return_quarterly, 2:3)
  gmm(esg_score esg_momentum value size profitability beta investment, 3:3)
  gmm(price_momentum, 4:5)
| timedumm collapse nolevel fod
"""

results_gmm_6 = regression.abond(command_gmm_6, df_model_split_2, ["stock", "t"])
print(results_gmm_6)

 Dynamic panel-data estimation, two-step difference GMM
 Group variable: stock                            Number of obs = 4598    
 Time variable: t                                 Min obs per group: 0    
 Number of instruments = 20                       Max obs per group: 9    
 Number of groups = 620                           Avg obs per group: 7.42 
+---------------------------+------------+---------------------+------------+-----------+-----+
|   stock_return_quarterly  |   coef.    | Corrected Std. Err. |     z      |   P>|z|   |     |
+---------------------------+------------+---------------------+------------+-----------+-----+
| L1.stock_return_quarterly | -0.0843161 |      0.1058470      | -0.7965848 | 0.4256922 |     |
|            beta           | 0.0341824  |      0.0413347      | 0.8269655  | 0.4082566 |     |
|            size           | 0.0252347  |      0.4212446      | 0.0599050  | 0.9522313 |     |
|           value           | -0.6563901 |      0.3744606      | -1.

#### GMM (7)

In [16]:
command_gmm_7 = """
stock_return_quarterly 
L(1:1).stock_return_quarterly
beta 
size 
value 
profitability 
investment
price_momentum 
esg_score 
esg_momentum 
| gmm(stock_return_quarterly, 2:3)
  gmm(esg_score esg_momentum value size profitability beta investment, 3:3)
  gmm(price_momentum, 4:5)
| timedumm collapse nolevel fod
"""

results_gmm_7 = regression.abond(command_gmm_7, df_model_split_3, ["stock", "t"])
print(results_gmm_7)

 Dynamic panel-data estimation, two-step difference GMM
 Group variable: stock                            Number of obs = 4093    
 Time variable: t                                 Min obs per group: 0    
 Number of instruments = 19                       Max obs per group: 8    
 Number of groups = 572                           Avg obs per group: 7.16 
+---------------------------+------------+---------------------+------------+-----------+-----+
|   stock_return_quarterly  |   coef.    | Corrected Std. Err. |     z      |   P>|z|   |     |
+---------------------------+------------+---------------------+------------+-----------+-----+
| L1.stock_return_quarterly | 0.0958727  |      0.1604541      | 0.5975082  | 0.5501681 |     |
|            beta           | 0.0049938  |      0.0132154      | 0.3778791  | 0.7055204 |     |
|            size           | -0.0118940 |      0.1689367      | -0.0704050 | 0.9438713 |     |
|           value           | -0.2211488 |      0.1419828      | -1.

#### Extracting to Excel

In [ ]:
def extract_gmm_results(
    gmm_result,
    model_name="GMM",
    variable_labels=None,
):
    """
    Extract coefficients, std. errors, z-stats,
    p-values, significance stars, Hansen p-value, AR(1), AR(2),
    and instrument count from a pydynpd regression.abond result.
    """

    model = gmm_result.models[0]

    # Main coefficient table
    tab = model.regression_table.copy()
    rename_map = {
        "variable": "variable",
        "coefficient": "coef",
        "std_err": "std_err",
        "z_value": "z_stat",
        "p_value": "p_value",
        "sig": "sig",
    }

    tab = tab.rename(columns={k: v for k, v in rename_map.items() if k in tab.columns})

    # Add missing columns
    for col in ["variable", "coef", "std_err", "z_stat", "p_value", "sig"]:
        if col not in tab.columns:
            tab[col] = np.nan

    # Significance stars
    def stars(p):
        if pd.isna(p):
            return ""
        if p < 0.01:
            return "***"
        if p < 0.05:
            return "**"
        if p < 0.10:
            return "*"
        return ""

    tab["stars"] = tab["p_value"].apply(stars)
    tab["coef_with_stars"] = tab["coef"].map(lambda x: f"{x:.4f}" if pd.notna(x) else "") + tab["stars"]
    tab["z_in_parentheses"] = tab["z_stat"].map(lambda x: f"({x:.2f})" if pd.notna(x) else "")

    if variable_labels is not None:
        tab["label"] = tab["variable"].map(variable_labels).fillna(tab["variable"])
    else:
        tab["label"] = tab["variable"]

    tab.insert(0, "model", model_name)

    hansen_p = getattr(getattr(model, "hansen", None), "p_value", np.nan)

    # Extract AR(1) and AR(2) p-values
    ar_list = getattr(model, "AR_list", []) or []
    ar1_p = next((ar.P_value for ar in ar_list if getattr(ar, "lag", None) == 1), np.nan)
    ar2_p = next((ar.P_value for ar in ar_list if getattr(ar, "lag", None) == 2), np.nan)

    instrument_count = getattr(model, "num_instr", np.nan)
    n_obs = getattr(model, "N", np.nan)

    tests = pd.DataFrame([{
        "model": model_name,
        "hansen_p": hansen_p,
        "ar1_p": ar1_p,
        "ar2_p": ar2_p,
        "instrument_count": instrument_count,
        "n_obs": n_obs,
    }])

    return tab, tests

In [ ]:
def export_gmm_results_to_excel(results_dict, output_path, time_col="t", variable_labels=None):
    all_tables = []
    all_tests = []

    for model_name, res in results_dict.items():
        tab, test = extract_gmm_results(
            res,
            model_name=model_name,
            time_col=time_col,
            variable_labels=variable_labels,
            keep_time_dummies=True,
        )
        all_tables.append(tab)
        all_tests.append(test)

    coef_long = pd.concat(all_tables, ignore_index=True)
    tests = pd.concat(all_tests, ignore_index=True)

    rows = []

    # Create rows for coefficients and z-stats in latex format (Easier to copy into overleaf than manually copying it one by one)
    for var in coef_long["label"].drop_duplicates():
        sub = coef_long[coef_long["label"] == var]

        coef_row = {"variable": var}
        z_row = {"variable": ""}

        for _, r in sub.iterrows():
            coef_row[r["model"]] = r["coef_with_stars"]
            z_row[r["model"]] = r["z_in_parentheses"]

        rows.append(coef_row)
        rows.append(z_row)

    appendix_table = pd.DataFrame(rows)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        coef_long.to_excel(writer, sheet_name="gmm_long", index=False)
        appendix_table.to_excel(writer, sheet_name="appendix_format", index=False)
        tests.to_excel(writer, sheet_name="tests", index=False)

    return coef_long, appendix_table, tests

In [20]:
results_dict = {
    "GMM 1": results_gmm_1,
    "GMM 2": results_gmm_2,
    "GMM 3": results_gmm_3,
    "GMM 4": results_gmm_4,
    "GMM 5": results_gmm_5,
    "GMM 6": results_gmm_6,
    "GMM 7": results_gmm_7,
}

output_path = path + "06_gmm_results.xlsx"

export_gmm_results_to_excel(results_dict, output_path=output_path)

(     model                   variable      coef   std_err    z_stat   p_value  \
 0    GMM 1  L1.stock_return_quarterly  0.016695  0.034689  0.481281  0.630317   
 1    GMM 1                       beta  0.005447  0.005798  0.939449  0.347501   
 2    GMM 1                       size  0.030551  0.040235  0.759324  0.447659   
 3    GMM 1                      value -0.081348  0.039110 -2.079961  0.037529   
 4    GMM 1              profitability  0.056621  0.037269  1.519262  0.128697   
 ..     ...                        ...       ...       ...       ...       ...   
 220  GMM 7                        t_8  0.085356  0.028790  2.964798  0.003029   
 221  GMM 7                        t_9 -0.033782  0.023098 -1.462571  0.143585   
 222  GMM 7                       t_10  0.028907  0.064581  0.447612  0.654433   
 223  GMM 7                       t_11  0.063231  0.025026  2.526669  0.011515   
 224  GMM 7                       t_12  0.005008  0.018498  0.270700  0.786621   
 
     sig stars